# 06｜LeRobot ACT + PushT：从数据集到训练、checkpoint、推理、评测与权重对比

**目标环境**

```text
Windows / Python 3.12 / LeRobot 0.6.1 / CUDA / RTX4060
Dataset: lerobot/pusht
```

这份 Notebook 把同一套流程分成两层：

```text
工程层：Dataset → 冒烟训练 → 正式训练 → checkpoint → 推理 → PushT闭环评测 → 社区权重比较
源码层：Dataset时间窗口 → DataLoader → Preprocessor → ACTPolicy.forward
      → L1/KL → backward → gradient clipping → AdamW → checkpoint
```

完成后你应该能独立回答：

- 有了 LeRobot Dataset 后 ACT 怎么训练？
- `lerobot-train` 内部调用哪些核心函数？
- 一次 training step 如何从 batch 变成参数更新？
- `loss`、`loss_dict`、L1、KL 各自是什么？
- checkpoint 为什么不只有 `model.safetensors`？
- 如何做离线推理和 PushT 闭环推理？
- 如何公平比较自己训练的 ACT 和社区 ACT？


## 0. 对比对象

本地模型：你自己从 `lerobot/pusht` 训练的 ACT。

参考模型：`aadarshram/act_pusht`。它是 Hugging Face Hub 上的**社区 ACT checkpoint**，模型卡说明使用 `lerobot/pusht` 训练约 80,000 steps。

截至本 Notebook 制作时，没有确认到 LeRobot 官方组织发布同任务的 `lerobot/act_pusht`。因此同架构对比时，社区 ACT 比拿 ACT 与官方 Diffusion Policy 比较更合适。


# Part A｜环境、版本、依赖


In [ ]:
from __future__ import annotations
import importlib.metadata, inspect, json, os, shutil, subprocess, sys, time
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any
import numpy as np
import torch
import torch.nn.functional as F
import lerobot

print('Python:', sys.executable)
print('Python version:', sys.version.split()[0])
print('LeRobot:', importlib.metadata.version('lerobot'))
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA build:', torch.version.cuda)
print('LeRobot package:', Path(lerobot.__file__).resolve())
print('HF_HOME:', os.environ.get('HF_HOME'))
print('HF_LEROBOT_HOME:', os.environ.get('HF_LEROBOT_HOME'))


In [ ]:
EXPECTED_LEROBOT = '0.6.1'
installed = importlib.metadata.version('lerobot')
print('版本匹配' if installed == EXPECTED_LEROBOT else f'警告：当前{installed}，本Notebook按{EXPECTED_LEROBOT}设计')


训练依赖缺失时：

```powershell
python -m pip install "lerobot[training]==0.6.1"
```

PushT 闭环环境缺失时：

```powershell
python -m pip install "lerobot[pusht]==0.6.1"
python -m pip check
```

安装后重启 Kernel。


In [ ]:
for module_name in ['accelerate','termcolor','tqdm','huggingface_hub']:
    try:
        m=__import__(module_name)
        print('[OK]', module_name, getattr(m,'__version__',''))
    except Exception as exc:
        print('[缺少]', module_name, exc)
try:
    import gym_pusht
    print('[OK] gym_pusht:', gym_pusht.__file__)
except Exception as exc:
    print('[PushT环境未就绪]', exc)


# Part B｜从 `lerobot/pusht` 数据集开始


In [ ]:
from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata
DATASET_ID='lerobot/pusht'
metadata=LeRobotDatasetMetadata(DATASET_ID)
print('repo_id:', metadata.repo_id)
print('root:', metadata.root)
print('fps:', metadata.fps)
print('episodes:', metadata.total_episodes)
print('frames:', metadata.total_frames)
print('\nfeatures:')
for k,v in metadata.features.items(): print(' ',k,v)


In [ ]:
single_dataset=LeRobotDataset(DATASET_ID)
sample=single_dataset[0]
for k,v in sample.items():
    print(f'{k:<28}', 'shape=',getattr(v,'shape',None), 'dtype=',getattr(v,'dtype',None))
print('\nstate:', sample['observation.state'])
print('expert action:', sample['action'])


PushT 中：

```text
observation.image  当前 RGB 画面
observation.state  agent二维位置
action             专家二维目标位置
```

普通 Dataset 返回单步 `action.shape=(2,)`；ACT 监督目标是未来动作块 `(chunk_size, 2)`。


# Part C｜Dataset Features → ACTConfig → ACTPolicy


In [ ]:
from lerobot.configs import FeatureType
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.utils.feature_utils import dataset_to_policy_features

policy_features=dataset_to_policy_features(metadata.features)
output_features={k:f for k,f in policy_features.items() if f.type is FeatureType.ACTION}
input_features={k:f for k,f in policy_features.items() if k not in output_features}
print('input_features:')
for k,v in input_features.items(): print(' ',k,v)
print('output_features:')
for k,v in output_features.items(): print(' ',k,v)


In [ ]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
act_cfg=ACTConfig(
    input_features=input_features,
    output_features=output_features,
    device=DEVICE,
    push_to_hub=False,
    chunk_size=100,
    n_action_steps=100,
    use_amp=False,
    vision_backbone='resnet18',
    use_vae=True,
    latent_dim=32,
    kl_weight=10.0,
)
print(act_cfg)


In [ ]:
policy=ACTPolicy(act_cfg).to(DEVICE)
print('Policy:', type(policy).__name__)
print('Network:', type(policy.model).__name__)
print('Device:', next(policy.parameters()).device)
print('Total parameters:', f'{sum(p.numel() for p in policy.parameters()):,}')
print('Trainable parameters:', f'{sum(p.numel() for p in policy.parameters() if p.requires_grad):,}')


# Part D｜action chunk：`delta_indices → delta_timestamps → Dataset`


`ACTConfig.action_delta_indices` 对 `chunk_size=100` 是 `[0,1,...,99]`。PushT `fps=10`，所以第 `i` 个未来动作相对时间是：

\[
\Delta t_i = i/fps
\]

例如第 20 帧就是 2.0 秒之后。


In [ ]:
print('前20个 indices:', act_cfg.action_delta_indices[:20])
print('长度:', len(act_cfg.action_delta_indices))


In [ ]:
def make_delta_timestamps(delta_indices, fps):
    if delta_indices is None: return [0.0]
    return [i/fps for i in delta_indices]

action_delta_timestamps=make_delta_timestamps(act_cfg.action_delta_indices, metadata.fps)
print('前20个 timestamps:', action_delta_timestamps[:20])
print('最后一个:', action_delta_timestamps[-1], 's')


In [ ]:
delta_timestamps={'action': action_delta_timestamps}
delta_timestamps |= {
    k: make_delta_timestamps(act_cfg.observation_delta_indices, metadata.fps)
    for k in act_cfg.image_features
}
act_dataset=LeRobotDataset(DATASET_ID, delta_timestamps=delta_timestamps)
act_sample=act_dataset[0]
print('普通action:', single_dataset[0]['action'].shape)
print('ACT action chunk:', act_sample['action'].shape)
print('action_is_pad:', act_sample['action_is_pad'].shape)


靠近 episode 结尾时未来动作不足 100 步，LeRobot 会 padding，并用 `action_is_pad` 标记无效位置；ACT 的 L1 loss 会忽略这些位置。


# Part E｜Preprocessor / Postprocessor


In [ ]:
from lerobot.policies import make_pre_post_processors
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=act_cfg,
    dataset_stats=metadata.stats,
    dataset_meta=metadata,
)
print('PREPROCESSOR\n', preprocessor)
print('\nPOSTPROCESSOR\n', postprocessor)


训练/推理输入链：

```text
Dataset Tensor → batching/normalization/device placement → ACT
```

推理输出链：

```text
normalized action → unnormalization → CPU / 环境动作尺度
```

因此 checkpoint 需要 processor 配置与归一化统计量，不能只保留模型权重。


# Part F｜DataLoader：得到真实训练 Batch


In [ ]:
from torch.utils.data import DataLoader
debug_loader=DataLoader(
    act_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE=='cuda'),
    drop_last=True,
)
raw_batch=next(iter(debug_loader))
for k in ['observation.image','observation.state','action','action_is_pad']:
    v=raw_batch.get(k)
    print(k, getattr(v,'shape',None), getattr(v,'dtype',None))


In [ ]:
processed_batch=preprocessor(raw_batch)
for k in ['observation.image','observation.state','action','action_is_pad']:
    v=processed_batch.get(k)
    if v is not None:
        print(k, 'shape=',getattr(v,'shape',None), 'dtype=',getattr(v,'dtype',None), 'device=',getattr(v,'device',None))


典型 batch：

```text
image          (B,3,H,W)
state          (B,2)
action         (B,100,2)
action_is_pad  (B,100)
```


# Part G｜直接运行 ACT `forward()`


In [ ]:
policy.train()
loss, loss_dict = policy.forward(processed_batch)
print('loss type:', type(loss), 'shape:', loss.shape)
print('loss:', loss)
print('loss_dict type:', type(loss_dict))
print('loss_dict:', loss_dict)


ACT `use_vae=True` 时：

\[
L=L_{L1}+\lambda_{KL}L_{KL}
\]

动作重建：

\[
L_{L1}=\mathrm{mean}_{valid}(|\hat a-a|)
\]

KL：

\[
L_{KL}=-\frac12\sum_j(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2)
\]

`loss` 是带计算图的标量 Tensor，用于 `backward()`；`loss_dict` 用于记录 `l1_loss`、`kld_loss` 等日志值。


# Part H｜把 `ACTPolicy.forward()` 的 L1 + KL 严格手工重算


In [ ]:
from lerobot.utils.constants import ACTION, OBS_IMAGES
model_batch=dict(processed_batch)
if policy.config.image_features:
    model_batch[OBS_IMAGES]=[model_batch[k] for k in policy.config.image_features]
policy.train()
actions_hat,(mu_hat,log_sigma_x2_hat)=policy.model(model_batch)
print('actions_hat:', actions_hat.shape)
print('expert action:', processed_batch[ACTION].shape)
print('mu:', None if mu_hat is None else mu_hat.shape)
print('log_sigma_x2:', None if log_sigma_x2_hat is None else log_sigma_x2_hat.shape)


In [ ]:
abs_err=F.l1_loss(processed_batch[ACTION], actions_hat, reduction='none')
valid_mask=~processed_batch['action_is_pad'].unsqueeze(-1)
num_valid=valid_mask.sum()*abs_err.shape[-1]
manual_l1=(abs_err*valid_mask).sum()/num_valid.clamp_min(1)
print('abs_err:', abs_err.shape)
print('valid_mask:', valid_mask.shape)
print('manual L1:', manual_l1.item())


In [ ]:
if mu_hat is not None and log_sigma_x2_hat is not None:
    kl_each=-0.5*(1+log_sigma_x2_hat-mu_hat.pow(2)-log_sigma_x2_hat.exp())
    manual_kld=kl_each.sum(dim=-1).mean()
    manual_total=manual_l1+act_cfg.kl_weight*manual_kld
    print('manual KLD:', manual_kld.item())
    print('kl_weight:', act_cfg.kl_weight)
    print('manual total:', manual_total.item())
else:
    manual_kld=None; manual_total=manual_l1
    print('未使用VAE')


训练模式中 VAE 每次会随机采样 `epsilon`，所以两次独立 forward 的预测值不要求完全一致；但 loss 公式一致。


# Part I｜latent `z`：把重参数化写成实际代码


In [ ]:
if mu_hat is not None:
    sigma_hat=torch.exp(0.5*log_sigma_x2_hat)
    epsilon=torch.randn_like(mu_hat)
    z_demo=mu_hat+sigma_hat*epsilon
    print('mu:', mu_hat.shape, 'sigma:',sigma_hat.shape, 'z:',z_demo.shape)
    print('sample0 mu[:8]:', mu_hat[0,:8].detach().cpu())
    print('sample0 sigma[:8]:', sigma_hat[0,:8].detach().cpu())
    print('sample0 z[:8]:', z_demo[0,:8].detach().cpu())


KL 不是把所有 `z` 训练成 0。它约束：

\[
q(z|o,a)=N(\mu,\sigma^2) \approx N(0,I)
\]

因此倾向于 `mu≈0`、`sigma²≈1`。训练时仍会采样不同 `z`；推理时没有专家 action，LeRobot ACT 直接用先验中心 `z=0` 获得确定性行为。


# Part J｜ACT 到底训练哪些 Parameter？


In [ ]:
groups=defaultdict(lambda:{'tensors':0,'numel':0})
for name,p in policy.named_parameters():
    if not p.requires_grad: continue
    if name.startswith('model.backbone'): g='ResNet backbone'
    elif 'vae' in name: g='CVAE / VAE'
    elif 'encoder' in name: g='Transformer Encoder'
    elif 'decoder' in name: g='Transformer Decoder'
    elif 'action_head' in name: g='Action Head'
    else: g='Projection / Embedding / Other'
    groups[g]['tensors']+=1; groups[g]['numel']+=p.numel()
for g,s in groups.items(): print(f'{g:<32} tensors={s["tensors"]:<4} params={s["numel"]:,}')


In [ ]:
shown=0
for name,p in policy.named_parameters():
    if p.requires_grad:
        print(f'{name:<78}', tuple(p.shape))
        shown+=1
        if shown>=50: break


最终学习的都是 `torch.nn.Parameter`：Linear 的 weight/bias、Attention 的 Q/K/V 和输出投影矩阵、ResNet 卷积核、LayerNorm 参数、Embedding、Action Head 等。


# Part K｜Optimizer：AdamW 与 ACT 参数组


In [ ]:
optim_groups=policy.get_optim_params()
print('参数组数量:', len(optim_groups))
for i,g in enumerate(optim_groups):
    print('group',i,'params=',f'{sum(p.numel() for p in g["params"]):,}','explicit lr=',g.get('lr','optimizer默认'))


In [ ]:
optimizer=torch.optim.AdamW(
    policy.get_optim_params(),
    lr=policy.config.optimizer_lr,
    weight_decay=policy.config.optimizer_weight_decay,
)
for i,g in enumerate(optimizer.param_groups):
    print('group',i,'lr=',g['lr'],'weight_decay=',g['weight_decay'],'tensor_count=',len(g['params']))


AdamW 直觉：`loss.backward()` 得到每个参数的梯度；AdamW 使用梯度的一阶/二阶历史统计、学习率和 weight decay 更新参数。ACT v0.6.1 默认 scheduler preset 为 `None`，所以标准 ACT 通常没有额外学习率调度。


# Part L｜手写一次完整 training step，看参数真的改变


In [ ]:
tracked_name,tracked_parameter=next((n,p) for n,p in policy.named_parameters() if p.requires_grad)
before=tracked_parameter.detach().clone()
print('跟踪参数:',tracked_name)
print('before:',before.flatten()[:5].cpu())


In [ ]:
policy.train()
optimizer.zero_grad(set_to_none=True)
loss,loss_dict=policy.forward(processed_batch)
loss.backward()
print('grad存在:',tracked_parameter.grad is not None)
if tracked_parameter.grad is not None:
    print('grad前5:',tracked_parameter.grad.flatten()[:5].detach().cpu())
grad_norm=torch.nn.utils.clip_grad_norm_(policy.parameters(),max_norm=10.0)
optimizer.step()
scheduler=None
if scheduler is not None: scheduler.step()
print('loss:',loss.item())
print('loss_dict:',loss_dict)
print('grad_norm:',float(grad_norm.detach().cpu()))


In [ ]:
after=tracked_parameter.detach().clone()
delta=(after-before).abs()
print('after:',after.flatten()[:5].cpu())
print('参数是否改变:',bool((delta>0).any()))
print('最大变化:',delta.max().item())
print('平均变化:',delta.mean().item())


一个 training step：

```text
batch → preprocessor → forward → loss → backward
→ gradient clipping → optimizer.step → 参数值变化
```

`steps=100000` 表示重复 100000 次参数更新，不是训练 100000 个 episode。


# Part M｜直接查看本机 LeRobot 0.6.1 的核心源码


In [ ]:
import lerobot.scripts.lerobot_train as train_module
print('train file:',inspect.getsourcefile(train_module))
print('ACT file:',inspect.getsourcefile(ACTPolicy))


In [ ]:
print('===== ACTPolicy.forward =====\n'); print(inspect.getsource(ACTPolicy.forward))


In [ ]:
print('===== ACTPolicy.select_action =====\n'); print(inspect.getsource(ACTPolicy.select_action))


In [ ]:
print('===== ACTPolicy.predict_action_chunk =====\n'); print(inspect.getsource(ACTPolicy.predict_action_chunk))


In [ ]:
print('===== update_policy =====\n'); print(inspect.getsource(train_module.update_policy))


阅读 `update_policy()` 重点找：

```text
policy.train()
accelerator.autocast()
policy.forward(batch)
accelerator.backward(loss)
clip_grad_norm_
optimizer.step()
optimizer.zero_grad()
lr_scheduler.step()  # 仅非None
```


# Part N｜官方 `train()` 总流程


In [ ]:
train_source=inspect.getsource(train_module.train).splitlines()
for i,line in enumerate(train_source[:280],start=1):
    print(f'{i:03d}: {line}')


官方调用图：

```text
CLI / TrainPipelineConfig
 ↓
make_train_eval_datasets(cfg)
 ↓
make_policy(cfg.policy, ds_meta=dataset.meta)
 ↓
make_pre_post_processors(...)
 ↓
make_optimizer_and_scheduler(cfg, policy)
 ↓
DataLoader(...)
 ↓
for step in range(cfg.steps):
    batch = next(...)
    batch = preprocessor(batch)
    update_policy(...)
    log / eval / save_checkpoint
```


# Part O｜冒烟训练：真实调用官方 `lerobot-train`


冒烟训练（smoke training）不追求性能，只验证整条训练链：数据、GPU、forward、loss、backward、optimizer、checkpoint 能不能完整跑通。


In [ ]:
PROJECT_ROOT=Path(r'D:\Desktop\robot\workspace\embodied_learning')
OUTPUT_ROOT=PROJECT_ROOT/'outputs'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run_stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
SMOKE_DIR=OUTPUT_ROOT/f'06_act_pusht_smoke_{run_stamp}'
train_exe=shutil.which('lerobot-train')
train_prefix=[train_exe] if train_exe else [sys.executable,'-m','lerobot.scripts.lerobot_train']
smoke_command=train_prefix+[
    '--dataset.repo_id=lerobot/pusht','--policy.type=act',
    f'--output_dir={SMOKE_DIR}','--job_name=06_act_pusht_smoke',
    f'--policy.device={DEVICE}','--policy.push_to_hub=false','--wandb.enable=false',
    '--batch_size=4','--num_workers=0','--steps=100','--log_freq=10',
    '--save_checkpoint=true','--save_freq=100','--env_eval_freq=0','--eval_steps=0',
    '--policy.chunk_size=100','--policy.n_action_steps=100','--policy.use_amp=false',
]
print('输出目录:',SMOKE_DIR)
print(subprocess.list2cmdline(smoke_command))


In [ ]:
RUN_SMOKE_TRAIN=False
if RUN_SMOKE_TRAIN:
    subprocess.run(smoke_command,cwd=PROJECT_ROOT,check=True)
else:
    print('未启动。确认前面单batch训练正常后，改为True。')


# Part P｜检查 checkpoint 文件结构


In [ ]:
def print_tree(root:Path,max_depth=6):
    if not root.exists(): print('目录不存在:',root); return
    root=root.resolve(); print(root.name+'/')
    for p in sorted(root.rglob('*')):
        rel=p.relative_to(root)
        if len(rel.parts)>max_depth: continue
        indent='    '*(len(rel.parts)-1); suffix='/' if p.is_dir() else ''
        print(f'{indent}├── {p.name}{suffix}')
print_tree(SMOKE_DIR)


In [ ]:
def find_pretrained_models(root):
    if not Path(root).exists(): return []
    return sorted(p for p in Path(root).rglob('pretrained_model') if p.is_dir())
smoke_model_dirs=find_pretrained_models(SMOKE_DIR)
for p in smoke_model_dirs: print(p)
LOCAL_SMOKE_MODEL=smoke_model_dirs[-1] if smoke_model_dirs else None
print('LOCAL_SMOKE_MODEL=',LOCAL_SMOKE_MODEL)


典型 checkpoint 会包含：

```text
pretrained_model/
├── config.json
├── model.safetensors
├── train_config.json
├── policy_preprocessor.json
├── policy_postprocessor.json
└── processor相关safetensors
```

`model.safetensors` 存网络 Parameter；processor 文件保证训练和推理使用相同归一化。


# Part Q｜正式训练（80k 作为第一版基线）


In [ ]:
full_stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
FULL_DIR=OUTPUT_ROOT/f'06_act_pusht_full_{full_stamp}'
full_command=train_prefix+[
    '--dataset.repo_id=lerobot/pusht','--policy.type=act',
    f'--output_dir={FULL_DIR}','--job_name=06_act_pusht_full',
    f'--policy.device={DEVICE}','--policy.push_to_hub=false','--wandb.enable=false',
    '--batch_size=8','--num_workers=0','--steps=80000','--log_freq=100',
    '--save_checkpoint=true','--save_freq=10000','--env_eval_freq=0','--eval_steps=0',
    '--policy.chunk_size=100','--policy.n_action_steps=100','--policy.use_amp=false',
]
print(subprocess.list2cmdline(full_command))


In [ ]:
RUN_FULL_TRAIN=False
if RUN_FULL_TRAIN:
    subprocess.run(full_command,cwd=PROJECT_ROOT,check=True)
else:
    print('正式训练未启动。')


# Part R｜从 checkpoint 恢复模型并推理


In [ ]:
from lerobot.policies import make_pre_post_processors

def load_act_checkpoint(model_path,device):
    model_path=str(model_path)
    p=ACTPolicy.from_pretrained(model_path).to(device)
    p.eval()
    pre,post=make_pre_post_processors(policy_cfg=p.config,pretrained_path=model_path)
    p.reset()
    return p,pre,post


In [ ]:
if LOCAL_SMOKE_MODEL is None:
    local_policy=local_pre=local_post=None
    print('尚无本地checkpoint')
else:
    local_policy,local_pre,local_post=load_act_checkpoint(LOCAL_SMOKE_MODEL,DEVICE)
    print('本地模型加载成功')
    print('chunk_size:',local_policy.config.chunk_size)
    print('n_action_steps:',local_policy.config.n_action_steps)


# Part S｜离线推理：一帧 observation → 一个 action


In [ ]:
def observation_from_sample(sample,policy):
    return {k:sample[k] for k in policy.config.input_features}

def infer_one_action(policy,preprocessor,postprocessor,sample):
    policy.reset()
    raw_obs=observation_from_sample(sample,policy)
    processed_obs=preprocessor(raw_obs)
    with torch.no_grad():
        normalized_action=policy.select_action(processed_obs)
    return postprocessor(normalized_action)


In [ ]:
if local_policy is None:
    print('先训练并加载本地checkpoint')
else:
    test_sample=single_dataset[0]
    pred=infer_one_action(local_policy,local_pre,local_post,test_sample)
    print('pred:',pred)
    print('expert:',test_sample['action'])


`predict_action_chunk()` 一次返回完整 `(B,chunk_size,action_dim)`；`select_action()` 每次返回一步，并管理内部 action queue。真实控制循环一般调用 `select_action()`。


In [ ]:
if local_policy is None:
    print('先加载本地checkpoint')
else:
    local_policy.reset()
    raw_obs=observation_from_sample(single_dataset[0],local_policy)
    processed_obs=local_pre(raw_obs)
    with torch.no_grad(): normalized_chunk=local_policy.predict_action_chunk(processed_obs)
    chunk=local_post(normalized_chunk)
    print('chunk shape:',chunk.shape)
    print('前5步:',chunk[0,:5])


# Part T｜加载社区 ACT 权重并查看仓库


In [ ]:
from huggingface_hub import HfApi
REFERENCE_MODEL_ID='aadarshram/act_pusht'
api=HfApi()
try:
    info=api.model_info(REFERENCE_MODEL_ID)
    print('模型存在:',REFERENCE_MODEL_ID,'last_modified=',info.last_modified)
    for f in api.list_repo_files(REFERENCE_MODEL_ID,repo_type='model'): print('├──',f)
except Exception as exc:
    print('Hub查询失败:',exc)


In [ ]:
RUN_LOAD_REFERENCE=False
if RUN_LOAD_REFERENCE:
    try:
        reference_policy,reference_pre,reference_post=load_act_checkpoint(REFERENCE_MODEL_ID,DEVICE)
        print('社区ACT加载成功')
        print('params:',f'{sum(p.numel() for p in reference_policy.parameters()):,}')
    except Exception as exc:
        reference_policy=reference_pre=reference_post=None
        print('加载失败:',type(exc).__name__,exc)
else:
    reference_policy=reference_pre=reference_post=None
    print('默认不自动下载约200MB模型。改为True后运行。')


# Part U｜先比较模型配置


In [ ]:
def summarize_act(name,p):
    c=p.config
    return dict(name=name,parameters=sum(x.numel() for x in p.parameters()),chunk_size=c.chunk_size,
                n_action_steps=c.n_action_steps,vision_backbone=c.vision_backbone,dim_model=c.dim_model,
                n_heads=c.n_heads,n_encoder_layers=c.n_encoder_layers,n_decoder_layers=c.n_decoder_layers,
                use_vae=c.use_vae,latent_dim=c.latent_dim,kl_weight=c.kl_weight)
for item in ([summarize_act('local',local_policy)] if local_policy else [])+([summarize_act(REFERENCE_MODEL_ID,reference_policy)] if reference_policy else []):
    print(json.dumps(item,indent=2,ensure_ascii=False))


比较前必须检查 `chunk_size / n_action_steps / backbone / Transformer规模 / VAE`。配置不同，结果差异就不只是“训练好坏”。


# Part V｜离线比较：同一批 observation 的第一步 action MAE


In [ ]:
def to_action_vector(x):
    x=x.detach().cpu() if torch.is_tensor(x) else torch.as_tensor(x)
    while x.ndim>1 and x.shape[0]==1: x=x.squeeze(0)
    return x.float()

def offline_action_mae(policy,pre,post,dataset,indices):
    errors=[]; rows=[]; policy.eval()
    for idx in indices:
        s=dataset[idx]
        pred=to_action_vector(infer_one_action(policy,pre,post,s))
        expert=to_action_vector(s['action'])
        mae=torch.mean(torch.abs(pred-expert)).item()
        errors.append(mae)
        rows.append(dict(index=idx,pred_x=float(pred[0]),pred_y=float(pred[1]),expert_x=float(expert[0]),expert_y=float(expert[1]),mae=mae))
    return dict(mean_mae=float(np.mean(errors)),std_mae=float(np.std(errors)),rows=rows)


In [ ]:
N_OFFLINE_SAMPLES=50
offline_indices=np.linspace(0,len(single_dataset)-1,N_OFFLINE_SAMPLES,dtype=int).tolist()
comparison={}
if local_policy is not None:
    comparison['local']=offline_action_mae(local_policy,local_pre,local_post,single_dataset,offline_indices)
if reference_policy is not None:
    comparison['reference']=offline_action_mae(reference_policy,reference_pre,reference_post,single_dataset,offline_indices)
for name,result in comparison.items():
    print(name,'mean MAE=',result['mean_mae'],'std=',result['std_mae'])


In [ ]:
for name,result in comparison.items():
    print('\n====',name,'====')
    for row in result['rows'][:10]: print(row)


离线 MAE 只是辅助指标，因为闭环中动作误差会改变下一时刻 observation，误差可能累积；最终必须看环境 rollout 成功率。


# Part W｜真正 PushT 闭环推理：`lerobot-eval`


官方闭环：

```text
make_env → reset → observation → env processor → policy preprocessor
→ select_action → policy postprocessor → env postprocessor → env.step
→ next observation → 循环
```

主要指标：`pc_success / avg_sum_reward / avg_max_reward`。


In [ ]:
eval_exe=shutil.which('lerobot-eval')
eval_prefix=[eval_exe] if eval_exe else [sys.executable,'-m','lerobot.scripts.lerobot_eval']
LOCAL_EVAL_DIR=OUTPUT_ROOT/'06_eval_local'
REFERENCE_EVAL_DIR=OUTPUT_ROOT/'06_eval_reference'
if LOCAL_SMOKE_MODEL is not None:
    local_eval_command=eval_prefix+[
        f'--policy.path={LOCAL_SMOKE_MODEL}','--env.type=pusht','--eval.n_episodes=10','--eval.batch_size=1',
        '--seed=42',f'--output_dir={LOCAL_EVAL_DIR}',f'--policy.device={DEVICE}','--policy.use_amp=false']
    print('LOCAL:',subprocess.list2cmdline(local_eval_command))
else:
    local_eval_command=None; print('尚无本地checkpoint')
reference_eval_command=eval_prefix+[
    f'--policy.path={REFERENCE_MODEL_ID}','--env.type=pusht','--eval.n_episodes=10','--eval.batch_size=1',
    '--seed=42',f'--output_dir={REFERENCE_EVAL_DIR}',f'--policy.device={DEVICE}','--policy.use_amp=false']
print('REFERENCE:',subprocess.list2cmdline(reference_eval_command))


In [ ]:
RUN_LOCAL_EVAL=False
if RUN_LOCAL_EVAL and local_eval_command is not None:
    subprocess.run(local_eval_command,cwd=PROJECT_ROOT,check=True)
else: print('本地闭环评测未启动')


In [ ]:
RUN_REFERENCE_EVAL=False
if RUN_REFERENCE_EVAL:
    subprocess.run(reference_eval_command,cwd=PROJECT_ROOT,check=True)
else: print('社区闭环评测未启动')


# Part X｜读取 `eval_info.json` 并比较


In [ ]:
def load_eval_overall(eval_dir):
    p=Path(eval_dir)/'eval_info.json'
    if not p.exists(): return None
    with open(p,'r',encoding='utf-8') as f: data=json.load(f)
    return data.get('overall',data)
local_eval=load_eval_overall(LOCAL_EVAL_DIR)
reference_eval=load_eval_overall(REFERENCE_EVAL_DIR)
print('local:',local_eval)
print('reference:',reference_eval)


推荐优先看：

```text
1 pc_success      任务完成率，最重要
2 avg_max_reward  episode最好状态
3 avg_sum_reward  整体trajectory奖励
4 offline MAE     辅助
5 inference latency / 显存 / 参数量  工程成本
```


# Part Y｜为什么 100-step 冒烟模型不能和 80k 社区模型直接谈优劣？


因为训练量完全不同。公平实验应尽量控制：

```text
本地 ACT 80k vs 社区 ACT 80k
相同 dataset/revision
相同 ACTConfig
相同 eval seed
相同 episode 数
相同 PushT 环境版本
```

100-step 模型只验证“训练管线通了”。


# Part Z｜官方 training step 的数学版展开


设模型参数为 \(\theta\)，batch 为 \(B\)。

1. 前向：
\[
\hat A=f_\theta(O,z)
\]

2. VAE 重参数化：
\[
z=\mu+\sigma\odot\epsilon,\quad \epsilon\sim N(0,I)
\]

3. 动作重建：
\[
L_{action}=\mathrm{mean}_{valid}|\hat A-A|
\]

4. KL：
\[
L_{KL}=-\frac12\sum_j(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2)
\]

5. 总损失：
\[
L=L_{action}+\beta L_{KL}
\]

6. 反向传播：
\[
g=\nabla_\theta L
\]

7. Gradient clipping：若 \(\|g\|>C\)，整体缩放。

8. AdamW 根据梯度、历史一阶/二阶统计、lr、weight decay 得到 \(\theta_{t+1}\)。


# Appendix 1｜教学版 `train_step`：与官方逻辑一一对应


In [ ]:
def educational_train_step(policy,raw_batch,preprocessor,optimizer,grad_clip_norm=10.0,scheduler=None):
    # 1) raw Dataset batch → normalize / batch / device
    batch=preprocessor(raw_batch)
    # 2) train mode
    policy.train()
    # 3) 清理旧梯度
    optimizer.zero_grad(set_to_none=True)
    # 4) forward: ACT预测 + L1/KL
    loss,loss_dict=policy.forward(batch)
    # 5) backward: 计算所有Parameter的 dLoss/dParameter
    loss.backward()
    # 6) gradient clipping
    grad_norm=torch.nn.utils.clip_grad_norm_(policy.parameters(),max_norm=grad_clip_norm)
    # 7) AdamW更新Parameter
    optimizer.step()
    # 8) 可选lr scheduler；ACT默认通常为None
    if scheduler is not None: scheduler.step()
    return {'loss':float(loss.detach().cpu()),'grad_norm':float(grad_norm.detach().cpu()),**loss_dict}

print(inspect.signature(educational_train_step))


# Appendix 2｜教学版推理闭环


In [ ]:
def conceptual_rollout_loop(env,policy,preprocessor,postprocessor,preprocess_env_observation,max_steps=300):
    policy.reset()  # 新episode清空action queue/temporal ensemble
    obs,info=env.reset()
    total_reward=0.0
    for step in range(max_steps):
        raw_policy_obs=preprocess_env_observation(obs)
        policy_input=preprocessor(raw_policy_obs)
        with torch.no_grad():
            action_normalized=policy.select_action(policy_input)
        action=postprocessor(action_normalized)
        obs,reward,terminated,truncated,info=env.step(action)
        total_reward+=float(reward)
        if terminated or truncated: break
    return total_reward,info


这段用于理解闭环结构，不直接替代 `lerobot-eval`；真实 PushT 评测还包含 vector env、env pre/post processor、seed、video 和 metric 汇总。


# Appendix 3｜本机核心源码路径


In [ ]:
LEROBOT_ROOT=Path(lerobot.__file__).resolve().parent
core_files=[
    LEROBOT_ROOT/'policies'/'act'/'configuration_act.py',
    LEROBOT_ROOT/'policies'/'act'/'modeling_act.py',
    LEROBOT_ROOT/'policies'/'act'/'processor_act.py',
    LEROBOT_ROOT/'policies'/'factory.py',
    LEROBOT_ROOT/'datasets'/'lerobot_dataset.py',
    LEROBOT_ROOT/'datasets'/'factory.py',
    LEROBOT_ROOT/'scripts'/'lerobot_train.py',
    LEROBOT_ROOT/'scripts'/'lerobot_eval.py',
    LEROBOT_ROOT/'optim'/'factory.py',
]
for p in core_files: print('[存在]' if p.exists() else '[不存在]',p)


# Appendix 4｜建议实际执行顺序


```text
1. 跑到 Part L
   确认单batch forward / loss / backward / 参数变化

2. RUN_SMOKE_TRAIN=True
   训练100 steps，确认官方trainer和checkpoint

3. 加载本地100-step checkpoint
   做离线推理，理解 from_pretrained/pre/select_action/post

4. RUN_LOAD_REFERENCE=True
   加载社区80k ACT，同样本做离线MAE

5. 安装 lerobot[pusht]
   RUN_LOCAL_EVAL / RUN_REFERENCE_EVAL
   比较 pc_success / reward

6. RUN_FULL_TRAIN=True
   本地训练80k

7. 本地80k vs 社区80k
   相同seed、相同episodes正式比较
```


# Appendix 5｜学习完成检查


- [ ] 普通 action `(2,)` 为什么变成 ACT `(100,2)`
- [ ] `delta_timestamps` 为什么除 fps
- [ ] `action_is_pad` 如何进入 L1
- [ ] pre/post processor 为什么必须随 checkpoint 保存
- [ ] `ACTPolicy.forward()` 为什么训练时需要 expert action
- [ ] L1/KL 各约束什么
- [ ] `mu / log_sigma_x2 / z` 的关系
- [ ] `loss.backward()` 生成什么
- [ ] `optimizer.step()` 修改什么
- [ ] `scheduler.step()` 与 optimizer 的区别
- [ ] 冒烟训练为什么不能用于性能结论
- [ ] `predict_action_chunk()` 与 `select_action()` 的区别
- [ ] 离线 MAE 为什么不能替代 closed-loop success
- [ ] 如何公平比较本地与社区 checkpoint
